In [2]:
import sys
from pathlib import Path
import json
from collections import Counter
import geopandas as gpd
import numpy as np

# adiciona a raiz do projeto ao Python
sys.path.append(str(Path.cwd().parent))

from src.domain.enums import IncomeGroup, TravelMode, TripPurpose
from src.domain.agent import Agent
from src.simulation.population import generate_population
from src.simulation.origin_assignment import assign_origins
from src.simulation.purpose_choice import assign_purpose


In [3]:
import geopandas as gpd
sys.path.append(str(Path.cwd().parent))

path = "../data/o-d/destinos_cnefe_porto_alegre.gpkg"

# Lista as layers do GeoPackage
print("Layers:")
print(gpd.list_layers(path))

# Lê a layer criada pelo destinationsCNEFE.py
gdf = gpd.read_file(
    path,
    layer="destinations_cnefe",
    engine="pyogrio"
)

print("\nQuantidade de registros:")
print(len(gdf))

print("\nCRS:")
print(gdf.crs)

print("\nColunas:")
print(gdf.columns.tolist())

print("\nEspécies CNEFE:")
print(
    gdf["cnefe_species"]
    .value_counts(dropna=False)
)

print("\nCategorias:")
print(
    gdf["category"]
    .value_counts(dropna=False)
)

print("\nPotencial de trabalho:")
print(
    gdf["work_potential"]
    .value_counts(dropna=False)
)

print("\nNível de geocodificação:")
print(
    gdf["NV_GEO_COORD"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPrimeiros registros:")

cols = [
    "destination_id",
    "COD_ESPECIE",
    "cnefe_species",
    "category",
    "work_potential",
    "NV_GEO_COORD",
    "geometry",
]

print(gdf[cols].head(10))

Layers:
                 name geometry_type
0  destinations_cnefe         Point

Quantidade de registros:
65331

CRS:
EPSG:31982

Colunas:
['COD_UNICO_ENDERECO', 'COD_UF', 'COD_MUNICIPIO', 'COD_DISTRITO', 'COD_SUBDISTRITO', 'COD_SETOR', 'NUM_QUADRA', 'NUM_FACE', 'CEP', 'DSC_LOCALIDADE', 'NOM_TIPO_SEGLOGR', 'NOM_TITULO_SEGLOGR', 'NOM_SEGLOGR', 'NUM_ENDERECO', 'DSC_MODIFICADOR', 'NOM_COMP_ELEM1', 'VAL_COMP_ELEM1', 'NOM_COMP_ELEM2', 'VAL_COMP_ELEM2', 'NOM_COMP_ELEM3', 'VAL_COMP_ELEM3', 'NOM_COMP_ELEM4', 'VAL_COMP_ELEM4', 'NOM_COMP_ELEM5', 'VAL_COMP_ELEM5', 'LATITUDE', 'LONGITUDE', 'NV_GEO_COORD', 'COD_ESPECIE', 'DSC_ESTABELECIMENTO', 'COD_INDICADOR_ESTAB_ENDERECO', 'COD_INDICADOR_CONST_ENDERECO', 'COD_INDICADOR_FINALIDADE_CONST', 'COD_TIPO_ESPECI', 'cnefe_species', 'category', 'work_potential', 'destination_id', 'geometry']

Espécies CNEFE:
cnefe_species
other_establishment           60530
religious                      1938
health                         1563
education                   

In [4]:
import pandas as pd

print("Preenchimento de DSC_ESTABELECIMENTO:")
print(
    gdf["DSC_ESTABELECIMENTO"]
    .isna()
    .value_counts()
)

print("\nExemplos:")
print(
    gdf["DSC_ESTABELECIMENTO"]
    .dropna()
    .sample(
        min(
            50,
            gdf["DSC_ESTABELECIMENTO"].notna().sum()
        ),
        random_state=42
    )
    .tolist()
)

Preenchimento de DSC_ESTABELECIMENTO:
DSC_ESTABELECIMENTO
False    65314
True        17
Name: count, dtype: int64

Exemplos:
['DETALHES DOCES CHOCOLATES', 'VAGO', 'FERRAGEM SOARES', 'EXPEDICAO CASA DAS RETROS LTDA', '2215', 'INDUSTRIA', 'MANHATAN MOTEL', 'VAGO', 'IMOBILIARIA', 'AERO LOG EMPRESA DE TRANSPORTES', 'MINI MERCADO', 'MULLER AUTOELETRICA', 'SALAS COMERCIAIS', 'MARIJUS CONFECCOES', 'LOJA VAGA', 'ARMARINHO DA DUDA', 'COMERCIO', 'DEPOSITO CANTINA', 'ESTACIONAMENTO', 'MULTIFIOS', 'SEM NOME FECHADO', 'CLINICA ODONTOLOGICA QUALIDENTE', 'VAGO', 'EMPRESA DE CONSERTO DE MAQUINA DE LAVA LOUCA', 'VAGO', 'LOJA DE ROUPAS', 'VAGO', 'BAR LANCHERIA DU CACIQUE', 'ALIVIO', 'SAPATARIA', 'MOBILIARE MOVEIS', 'BANCO BANRISUL', 'CENTRO PROFISSIONAL', 'SALAO DE FESTAS', 'FABRICA DE PORTAS E JANELAS SOB MEDIDA', 'OFICINA DE BICICLETAS', 'TRANSPORTADORA JOMI', 'MERCADO', 'S E B', 'GERACAO SAUDE', 'COSER ADVOCACIA E CONSULTORIA', 'MERCADO', 'ATELIE DE ARTESANATO', 'RESTAURANTE', 'VAGO', 'ADVOCACIA', 'D

In [5]:
print("Total de registros:")
print(len(gdf))

print("\nEndereços únicos:")
print(gdf["COD_UNICO_ENDERECO"].nunique())

print("\nQuantidade de registros por endereço:")
registros_por_endereco = (
    gdf.groupby("COD_UNICO_ENDERECO")
    .size()
)

print(registros_por_endereco.describe())

print("\nEndereços com mais de 1 estabelecimento:")
print((registros_por_endereco > 1).sum())

print("\nMaiores quantidades de estabelecimentos no mesmo endereço:")
print(
    registros_por_endereco
    .sort_values(ascending=False)
    .head(20)
)

Total de registros:
65331

Endereços únicos:
65002

Quantidade de registros por endereço:
count    65002.000000
mean         1.005061
std          0.072677
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          4.000000
dtype: float64

Endereços com mais de 1 estabelecimento:
322

Maiores quantidades de estabelecimentos no mesmo endereço:
COD_UNICO_ENDERECO
73516017    4
73336503    3
73315432    3
73315479    3
74849731    3
96427092    3
73397685    2
73503677    2
73314255    2
73314256    2
74074344    2
73315529    2
73382269    2
73367790    2
74199885    2
73382310    2
73314265    2
73397698    2
73397699    2
73315149    2
dtype: int64


In [6]:
# Identifica endereços com mais de um registro
enderecos_duplicados = (
    gdf.groupby("COD_UNICO_ENDERECO")
    .size()
)

enderecos_duplicados = enderecos_duplicados[
    enderecos_duplicados > 1
].index

duplicados = gdf[
    gdf["COD_UNICO_ENDERECO"].isin(enderecos_duplicados)
].copy()

cols = [
    "COD_UNICO_ENDERECO",
    "NOM_SEGLOGR",
    "NUM_ENDERECO",
    "COD_ESPECIE",
    "cnefe_species",
    "DSC_ESTABELECIMENTO",
    "COD_INDICADOR_ESTAB_ENDERECO",
]

print("Distribuição do indicador entre os endereços duplicados:")
print(
    duplicados["COD_INDICADOR_ESTAB_ENDERECO"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nExemplos de endereços com múltiplos registros:")
print(
    duplicados[cols]
    .sort_values(
        ["COD_UNICO_ENDERECO", "DSC_ESTABELECIMENTO"]
    )
    .head(40)
    .to_string(index=False)
)

Distribuição do indicador entre os endereços duplicados:
COD_INDICADOR_ESTAB_ENDERECO
1.0    215
2.0     96
3.0     73
4.0    267
Name: count, dtype: int64

Exemplos de endereços com múltiplos registros:
 COD_UNICO_ENDERECO        NOM_SEGLOGR  NUM_ENDERECO  COD_ESPECIE       cnefe_species                      DSC_ESTABELECIMENTO  COD_INDICADOR_ESTAB_ENDERECO
           73314159             CAMARA           406            5              health          CLUNICA D3 OBESIDADE JOAO VIDAL                           4.0
           73314159             CAMARA           406            6 other_establishment                             VARIAS SALAS                           3.0
           73314255      ANDRADE NEVES           155            5              health                             VARIAS SALAS                           2.0
           73314255      ANDRADE NEVES           155            6 other_establishment                             VARIAS SALAS                           4.0
           

In [7]:
print("Valores de COD_TIPO_ESPECI:")
print(
    gdf["COD_TIPO_ESPECI"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nRelação entre COD_ESPECIE e COD_TIPO_ESPECI:")
print(
    pd.crosstab(
        gdf["COD_ESPECIE"],
        gdf["COD_TIPO_ESPECI"],
        dropna=False
    )
)

print("\nExemplos por COD_TIPO_ESPECI:")

amostra_tipo = (
    gdf[
        [
            "COD_TIPO_ESPECI",
            "COD_ESPECIE",
            "cnefe_species",
            "DSC_ESTABELECIMENTO",
        ]
    ]
    .dropna(subset=["COD_TIPO_ESPECI"])
    .sort_values("COD_TIPO_ESPECI")
)

for tipo, grupo in amostra_tipo.groupby("COD_TIPO_ESPECI"):
    print(f"\n--- COD_TIPO_ESPECI = {tipo} ---")
    print(
        grupo["DSC_ESTABELECIMENTO"]
        .dropna()
        .head(15)
        .tolist()
    )

Valores de COD_TIPO_ESPECI:
COD_TIPO_ESPECI
NaN    65331
Name: count, dtype: int64

Relação entre COD_ESPECIE e COD_TIPO_ESPECI:
COD_TIPO_ESPECI    NaN
COD_ESPECIE           
3                  128
4                 1172
5                 1563
6                60530
8                 1938

Exemplos por COD_TIPO_ESPECI:


In [8]:
other = gdf[
    gdf["cnefe_species"] == "other_establishment"
].copy()

print("Total de other_establishment:")
print(len(other))

print("\nDescrições únicas:")
print(other["DSC_ESTABELECIMENTO"].nunique())

print("\n20 descrições mais frequentes:")
print(
    other["DSC_ESTABELECIMENTO"]
    .str.strip()
    .str.upper()
    .value_counts(dropna=False)
    .head(20)
)

print("\n50 descrições mais frequentes:")
freq = (
    other["DSC_ESTABELECIMENTO"]
    .str.strip()
    .str.upper()
    .value_counts(dropna=False)
    .head(50)
)

print(freq.to_string())

Total de other_establishment:
60530

Descrições únicas:
37741

20 descrições mais frequentes:
DSC_ESTABELECIMENTO
VAGO                       4203
ESTACIONAMENTO              698
DEPOSITO                    671
BAR                         513
GARAGEM                     493
LOJA VAGA                   423
OFICINA MECANICA            389
SEM NOME                    361
SALAO DE BELEZA             346
ESCRITORIO DE ADVOCACIA     226
PREDIO COMERCIAL            223
BARBEARIA                   212
MINI MERCADO                211
ESCRITORIO                  209
OFICINA                     203
LOJA                        194
BRECHO                      193
LANCHERIA                   176
RESTAURANTE                 172
SERRALHERIA                 169
Name: count, dtype: int64

50 descrições mais frequentes:
DSC_ESTABELECIMENTO
VAGO                           4203
ESTACIONAMENTO                  698
DEPOSITO                        671
BAR                             513
GARAGEM                 

In [9]:
import re

other = gdf[
    gdf["cnefe_species"] == "other_establishment"
].copy()

# Normalização básica da descrição
other["descricao_norm"] = (
    other["DSC_ESTABELECIMENTO"]
    .fillna("")
    .str.upper()
    .str.strip()
)

# Casos claramente sem atividade identificável
padrao_inativo = (
    r"\bVAGO\b|"
    r"\bVAGA\b|"
    r"\bFECHADO\b|"
    r"\bSEM NOME\b"
)

other["sem_atividade_identificavel"] = (
    other["descricao_norm"]
    .str.contains(
        padrao_inativo,
        regex=True,
        na=False
    )
)

print("Total de other_establishment:")
print(len(other))

print("\nSem atividade identificável:")
print(
    other["sem_atividade_identificavel"]
    .value_counts()
)

print("\nPercentual:")
print(
    (
        other["sem_atividade_identificavel"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

print("\nDescrições mais frequentes marcadas:")
print(
    other.loc[
        other["sem_atividade_identificavel"],
        "descricao_norm"
    ]
    .value_counts()
    .head(30)
    .to_string()
)

Total de other_establishment:
60530

Sem atividade identificável:
sem_atividade_identificavel
False    54501
True      6029
Name: count, dtype: int64

Percentual:
sem_atividade_identificavel
False    90.04
True      9.96
Name: proportion, dtype: float64

Descrições mais frequentes marcadas:
descricao_norm
VAGO                       4203
LOJA VAGA                   423
SEM NOME                    361
ESTABELECIMENTO VAGO        161
FECHADO                     158
DEPOSITO VAGO                76
COMERCIAL VAGO               27
SALA COMERCIAL VAGA          27
VAGA                         25
BAR SEM NOME                 24
LOJA VAGO                    24
PREDIO VAGO                  22
PAVILHAO VAGO                12
COMERCIO VAGO                12
COML VAGO                    11
PREDIO COMERCIAL VAGO        11
TERRENO VAGO                  8
LJ VAGA                       8
ESTABELECIMENTO FECHADO       7
DEPOSITO SEM NOME             7
SALA VAGA                     7
SALA COMERCIAL VAGO  

In [10]:
other = gdf[
    gdf["cnefe_species"] == "other_establishment"
].copy()

other["descricao_norm"] = (
    other["DSC_ESTABELECIMENTO"]
    .fillna("")
    .str.upper()
    .str.strip()
)

# 1. Claramente inativos
other["inativo"] = (
    other["descricao_norm"]
    .str.contains(
        r"\bVAGO\b|\bVAGA\b|\bFECHADO\b",
        regex=True,
        na=False
    )
)

# 2. Apenas "SEM NOME", sem nenhuma outra informação
other["sem_nome_apenas"] = (
    other["descricao_norm"] == "SEM NOME"
)

# 3. Contém "SEM NOME", mas também contém informação funcional
other["sem_nome_com_descricao"] = (
    other["descricao_norm"]
    .str.contains(
        r"\bSEM NOME\b",
        regex=True,
        na=False
    )
    & ~other["sem_nome_apenas"]
    & ~other["inativo"]
)

print("Total de other_establishment:")
print(len(other))

print("\nClaramente inativos:")
print(other["inativo"].sum())

print("\nSomente 'SEM NOME':")
print(other["sem_nome_apenas"].sum())

print("\n'SEM NOME' mas com descrição adicional:")
print(other["sem_nome_com_descricao"].sum())

print("\nExemplos de 'SEM NOME' com informação funcional:")
print(
    other.loc[
        other["sem_nome_com_descricao"],
        "descricao_norm"
    ]
    .value_counts()
    .head(30)
    .to_string()
)

print("\nTotal realmente sem atividade utilizável neste critério:")
total_excluir = (
    other["inativo"]
    | other["sem_nome_apenas"]
)

print(total_excluir.sum())

print("\nPercentual:")
print(
    round(
        total_excluir.mean() * 100,
        2
    )
)

Total de other_establishment:
60530

Claramente inativos:
5497

Somente 'SEM NOME':
361

'SEM NOME' mas com descrição adicional:
171

Exemplos de 'SEM NOME' com informação funcional:
descricao_norm
BAR SEM NOME                     24
DEPOSITO SEM NOME                 7
ESTACIONAMENTO SEM NOME           6
BRECHO SEM NOME                   5
SEM NOME DEPOSITO                 3
SEM NOME VISIVEL                  3
RESTAURANTE SEM NOME              3
GARAGEM SEM NOME                  3
OFICINA SEM NOME                  3
LOJA SEM NOME                     3
SEM NOME PARA ALUGAR              3
MERCADINHO SEM NOME               3
MECANICA SEM NOME                 2
SEM NOME ESTACIONAMENTO           2
MINI MERCADO SEM NOME             2
PREDIO COMERCIAL SEM NOME         2
SEM NOME COMERCIO DESATIVADO      2
SEM NOME SALAO DE BELEZA          2
BARBEARIA SEM NOME                2
ARMAZEM SEM NOME                  2
MINIMERCADO SEM NOME              2
LANCHERIA SEM NOME                2
EM REFORMA

In [11]:
termos_estado = [
    "DESATIVADO",
    "DESATIVADA",
    "DESOCUPADO",
    "DESOCUPADA",
    "INATIVO",
    "INATIVA",
    "ABANDONADO",
    "ABANDONADA",
    "PARA ALUGAR",
    "ALUGA",
    "ALUGUEL",
    "EM REFORMA",
    "REFORMA",
    "DEMOLIDO",
    "DEMOLIDA",
    "VAZIO",
    "VAZIA",
]

print("Ocorrências de possíveis estados de inatividade:\n")

for termo in termos_estado:
    mascara = other["descricao_norm"].str.contains(
        termo,
        regex=False,
        na=False
    )

    quantidade = mascara.sum()

    if quantidade > 0:
        print(f"{termo}: {quantidade}")


# Junta todos os termos para inspecionar os casos encontrados
padrao_estado = "|".join(termos_estado)

suspeitos = other[
    other["descricao_norm"].str.contains(
        padrao_estado,
        regex=True,
        na=False
    )
].copy()

print("\nTotal de registros encontrados:")
print(len(suspeitos))

print("\nDescrições mais frequentes:")
print(
    suspeitos["descricao_norm"]
    .value_counts()
    .head(50)
    .to_string()
)

Ocorrências de possíveis estados de inatividade:

DESATIVADO: 28
DESATIVADA: 11
DESOCUPADO: 52
DESOCUPADA: 18
INATIVO: 3
ABANDONADO: 21
ABANDONADA: 7
PARA ALUGAR: 81
ALUGA: 139
ALUGUEL: 61
EM REFORMA: 11
REFORMA: 58
DEMOLIDO: 11
DEMOLIDA: 1
VAZIO: 68
VAZIA: 34

Total de registros encontrados:
505

Descrições mais frequentes:
descricao_norm
VAZIO                           51
DESOCUPADO                      36
PARA ALUGAR                     21
LOJA PARA ALUGAR                20
ALUGA                           18
LOJA VAZIA                      18
LOJA DESOCUPADA                 10
ESTABELECIMENTO ABANDONADO       7
ALUGUEL                          7
ALUGAR                           6
DEMOLIDORA                       6
TERRENO VAZIO                    5
ALUGASE                          4
EM REFORMA                       4
CASA DE ALUGUEL VAGA             4
SALA VAZIA                       4
IMOVEL DESOCUPADO                4
DEPOSITO DESATIVADO              4
DESATIVADO                  

In [12]:
import re

# Estados que indicam claramente ausência de atividade no endereço
padrao_inativo_refinado = (
    r"\bVAGO\b|"
    r"\bVAGA\b|"
    r"\bVAZIO\b|"
    r"\bVAZIA\b|"
    r"\bDESOCUPADO\b|"
    r"\bDESOCUPADA\b|"
    r"\bDESATIVADO\b|"
    r"\bDESATIVADA\b|"
    r"\bINATIVO\b|"
    r"\bINATIVA\b|"
    r"\bABANDONADO\b|"
    r"\bABANDONADA\b|"
    r"\bDEMOLIDO\b|"
    r"\bDEMOLIDA\b|"
    r"\bPARA\s+ALUGAR\b|"
    r"\bP\s+ALUGAR\b|"
    r"\bP/\s*ALUGAR\b|"
    r"\bALUGA[\s-]*SE\b|"
    r"\bEM\s+REFORMA\b"
)

other["inativo_refinado"] = (
    other["descricao_norm"]
    .str.contains(
        padrao_inativo_refinado,
        regex=True,
        na=False
    )
)

# "SEM NOME" sozinho continua sem informação funcional
other["sem_informacao_funcional"] = (
    other["descricao_norm"] == "SEM NOME"
)

other["excluir_destino"] = (
    other["inativo_refinado"]
    | other["sem_informacao_funcional"]
)

print("Total de other_establishment:")
print(len(other))

print("\nMarcados como inativos:")
print(other["inativo_refinado"].sum())

print("\nSomente 'SEM NOME':")
print(other["sem_informacao_funcional"].sum())

print("\nTotal a excluir:")
print(other["excluir_destino"].sum())

print("\nPercentual a excluir:")
print(
    round(
        other["excluir_destino"].mean() * 100,
        2
    )
)

print("\nDescrições mais frequentes excluídas:")
print(
    other.loc[
        other["excluir_destino"],
        "descricao_norm"
    ]
    .value_counts()
    .head(40)
    .to_string()
)

print("\nDescrições contendo REFORMA que NÃO foram excluídas:")
print(
    other.loc[
        other["descricao_norm"].str.contains(
            r"\bREFORMA\b",
            regex=True,
            na=False
        )
        & ~other["excluir_destino"],
        "descricao_norm"
    ]
    .value_counts()
    .head(20)
    .to_string()
)

Total de other_establishment:
60530

Marcados como inativos:
5601

Somente 'SEM NOME':
361

Total a excluir:
5962

Percentual a excluir:
9.85

Descrições mais frequentes excluídas:
descricao_norm
VAGO                          4203
LOJA VAGA                      423
SEM NOME                       361
ESTABELECIMENTO VAGO           161
DEPOSITO VAGO                   76
VAZIO                           51
DESOCUPADO                      36
COMERCIAL VAGO                  27
SALA COMERCIAL VAGA             27
VAGA                            25
LOJA VAGO                       24
PREDIO VAGO                     22
PARA ALUGAR                     21
LOJA PARA ALUGAR                20
LOJA VAZIA                      18
PAVILHAO VAGO                   12
COMERCIO VAGO                   12
COML VAGO                       11
PREDIO COMERCIAL VAGO           11
LOJA DESOCUPADA                 10
TERRENO VAGO                     8
LJ VAGA                          8
ESTABELECIMENTO ABANDONADO       7

In [13]:
import re
import unicodedata


def normalizar_descricao(texto):
    if texto is None:
        return ""

    texto = str(texto).upper().strip()

    # Remove acentos
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        char for char in texto
        if not unicodedata.combining(char)
    )

    # Mantém apenas letras, números e espaços
    texto = re.sub(
        r"[^A-Z0-9\s]",
        " ",
        texto
    )

    # Remove espaços duplicados
    texto = re.sub(
        r"\s+",
        " ",
        texto
    ).strip()

    return texto


# Mantém apenas os registros utilizáveis
other_ativos = other[
    ~other["excluir_destino"]
].copy()

other_ativos["descricao_classificacao"] = (
    other_ativos["DSC_ESTABELECIMENTO"]
    .apply(normalizar_descricao)
)

print("Registros ativos:")
print(len(other_ativos))

print("\nDescrições originais x normalizadas:")
print(
    other_ativos[
        [
            "DSC_ESTABELECIMENTO",
            "descricao_classificacao"
        ]
    ]
    .sample(
        min(30, len(other_ativos)),
        random_state=42
    )
    .to_string(index=False)
)

print("\n30 descrições normalizadas mais frequentes:")
print(
    other_ativos["descricao_classificacao"]
    .value_counts()
    .head(30)
    .to_string()
)

Registros ativos:
54568

Descrições originais x normalizadas:
                                 DSC_ESTABELECIMENTO                              descricao_classificacao
                                           FUNILARIA                                            FUNILARIA
                               CRECHE ESPACO BRINCAR                                CRECHE ESPACO BRINCAR
                             TODAS AS RACAS PET SHOP                              TODAS AS RACAS PET SHOP
                                                  SN                                                   SN
                                       AS RECICLADOS                                        AS RECICLADOS
                                              GALPAO                                               GALPAO
                           SOLTEC COMERCIAL ELETRICA                            SOLTEC COMERCIAL ELETRICA
                                      TOCA DO MUSICO                                      

In [14]:
termos_auditoria = [
    "FECHADO",
    "FECHADA",
    "SN",
    "S N",
    "SEM IDENTIFICACAO",
    "SEM INFORMACAO",
    "NAO IDENTIFICADO",
    "NAO IDENTIFICADA",
    "NAO INFORMADO",
    "NAO INFORMADA",
    "DESCONHECIDO",
    "DESCONHECIDA",
    "IGNORADO",
    "IGNORADA",
]

print("Ocorrências exatas entre os registros considerados ativos:\n")

for termo in termos_auditoria:
    quantidade = (
        other_ativos["descricao_classificacao"]
        == termo
    ).sum()

    if quantidade > 0:
        print(f"{termo}: {quantidade}")


print("\nDescrições contendo 'FECHAD':")
print(
    other_ativos.loc[
        other_ativos["descricao_classificacao"]
        .str.contains(
            "FECHAD",
            regex=False,
            na=False
        ),
        "descricao_classificacao"
    ]
    .value_counts()
    .head(30)
    .to_string()
)


print("\nDescrições contendo 'SN':")
print(
    other_ativos.loc[
        other_ativos["descricao_classificacao"]
        .str.contains(
            r"\bSN\b",
            regex=True,
            na=False
        ),
        "descricao_classificacao"
    ]
    .value_counts()
    .head(30)
    .to_string()
)

Ocorrências exatas entre os registros considerados ativos:

FECHADO: 158
FECHADA: 1
SN: 38
S N: 1
SEM IDENTIFICACAO: 46
NAO IDENTIFICADO: 9
NAO IDENTIFICADA: 1
NAO INFORMADO: 5
DESCONHECIDO: 5

Descrições contendo 'FECHAD':
descricao_classificacao
FECHADO                                 158
LOJA FECHADA                             22
ESTABELECIMENTO FECHADO                   7
COMERCIO FECHADO                          6
LOJA DE FECHADURAS                        6
PREDIO FECHADO                            4
MERCADO FECHADO                           4
GALPAO FECHADO                            4
ESTABELECIMENTO COMERCIAL FECHADO         3
SEM NOME FECHADO                          3
BAR FECHADO                               3
DEPOSITO FECHADO                          2
LOJA COMERCIAL FECHADA                    2
TERRENO FECHADO                           2
FABRICA FECHADA                           2
SALA COMERCIAL FECHADO                    1
LAN HOUSE FECHADA                         1
FECH

In [15]:
# Acrescenta "FECHADO/FECHADA" à regra de inatividade,
# sem confundir com palavras como "FECHADURAS"

padrao_fechado = r"\bFECHADO\b|\bFECHADA\b"

other["inativo_v2"] = (
    other["inativo_refinado"]
    | other["descricao_norm"].str.contains(
        padrao_fechado,
        regex=True,
        na=False
    )
)

# Descrições que não trazem informação funcional quando aparecem sozinhas
sem_informacao_exata = {
    "SEM NOME",
    "SN",
    "S N",
    "SEM IDENTIFICACAO",
    "SEM INFORMACAO",
    "NAO IDENTIFICADO",
    "NAO IDENTIFICADA",
    "NAO INFORMADO",
    "NAO INFORMADA",
    "DESCONHECIDO",
    "DESCONHECIDA",
    "IGNORADO",
    "IGNORADA",
}

other["sem_informacao_v2"] = (
    other["descricao_norm"].isin(
        sem_informacao_exata
    )
)

other["excluir_destino_v2"] = (
    other["inativo_v2"]
    | other["sem_informacao_v2"]
)

print("Total de other_establishment:")
print(len(other))

print("\nTotal a excluir com a regra anterior:")
print(other["excluir_destino"].sum())

print("\nTotal a excluir com a regra nova:")
print(other["excluir_destino_v2"].sum())

print("\nNovos registros excluídos:")
novos = (
    other["excluir_destino_v2"]
    & ~other["excluir_destino"]
)

print(novos.sum())

print("\nDescrições mais frequentes entre os novos excluídos:")
print(
    other.loc[
        novos,
        "descricao_norm"
    ]
    .value_counts()
    .head(40)
    .to_string()
)

print("\nVerificação de 'FECHADURAS':")
print(
    other.loc[
        other["descricao_norm"].str.contains(
            "FECHADUR",
            regex=False,
            na=False
        ),
        [
            "descricao_norm",
            "excluir_destino_v2"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\nRegistros restantes após a nova regra:")
print(
    (~other["excluir_destino_v2"]).sum()
)

Total de other_establishment:
60530

Total a excluir com a regra anterior:
5962

Total a excluir com a regra nova:
6343

Novos registros excluídos:
381

Descrições mais frequentes entre os novos excluídos:
descricao_norm
FECHADO                                 158
SEM IDENTIFICACAO                        46
SN                                       38
LOJA FECHADA                             22
NAO IDENTIFICADO                          9
ESTABELECIMENTO FECHADO                   7
COMERCIO FECHADO                          6
NAO INFORMADO                             5
DESCONHECIDO                              5
PREDIO FECHADO                            4
MERCADO FECHADO                           4
GALPAO FECHADO                            4
ESTABELECIMENTO COMERCIAL FECHADO         3
SEM NOME FECHADO                          3
BAR FECHADO                               3
DEPOSITO FECHADO                          2
LOJA COMERCIAL FECHADA                    2
TERRENO FECHADO                

In [16]:
# Conjunto definitivo de other_establishment ativos
other_ativos = other[
    ~other["excluir_destino_v2"]
].copy()

# Garante a descrição normalizada
other_ativos["descricao_classificacao"] = (
    other_ativos["DSC_ESTABELECIMENTO"]
    .apply(normalizar_descricao)
)

# Termos inicialmente associados a comércio/varejo
termos_shopping = [
    "LOJA",
    "MERCADO",
    "MINI MERCADO",
    "MINIMERCADO",
    "MERCADINHO",
    "SUPERMERCADO",
    "HIPERMERCADO",
    "BRECHO",
    "BAZAR",
    "ARMARINHO",
    "FERRAGEM",
    "LIVRARIA",
    "PAPELARIA",
    "FLORICULTURA",
    "SAPATARIA",
    "CALCADOS",
    "ROUPAS",
    "CONFECCOES",
    "MOVEIS",
    "ELETRODOMESTICOS",
    "MATERIAL DE CONSTRUCAO",
    "PET SHOP",
]

print("Total de registros ativos:")
print(len(other_ativos))

print("\nOcorrências por termo candidato a shopping:\n")

for termo in termos_shopping:
    mascara = (
        other_ativos["descricao_classificacao"]
        .str.contains(
            rf"\b{termo}\b",
            regex=True,
            na=False
        )
    )

    quantidade = mascara.sum()

    if quantidade > 0:
        print(f"{termo}: {quantidade}")


# Une todos os termos somente para inspeção
padrao_shopping = "|".join(
    rf"\b{termo}\b"
    for termo in termos_shopping
)

candidatos_shopping = other_ativos[
    other_ativos["descricao_classificacao"]
    .str.contains(
        padrao_shopping,
        regex=True,
        na=False
    )
].copy()

print("\nTotal de registros candidatos a shopping:")
print(len(candidatos_shopping))

print("\n50 descrições mais frequentes entre os candidatos:")
print(
    candidatos_shopping[
        "descricao_classificacao"
    ]
    .value_counts()
    .head(50)
    .to_string()
)

print("\nAmostra aleatória de 50 candidatos:")
print(
    candidatos_shopping[
        "descricao_classificacao"
    ]
    .sample(
        min(50, len(candidatos_shopping)),
        random_state=42
    )
    .tolist()
)

Total de registros ativos:
54187

Ocorrências por termo candidato a shopping:

LOJA: 1793
MERCADO: 1531
MINI MERCADO: 570
MINIMERCADO: 134
MERCADINHO: 120
SUPERMERCADO: 176
HIPERMERCADO: 2
BRECHO: 488
BAZAR: 479
ARMARINHO: 9
FERRAGEM: 550
LIVRARIA: 60
PAPELARIA: 71
FLORICULTURA: 114
SAPATARIA: 110
CALCADOS: 72
ROUPAS: 351
CONFECCOES: 74
MOVEIS: 398
ELETRODOMESTICOS: 28
MATERIAL DE CONSTRUCAO: 27
PET SHOP: 204

Total de registros candidatos a shopping:
6291

50 descrições mais frequentes entre os candidatos:
descricao_classificacao
MINI MERCADO                          211
LOJA                                  194
BRECHO                                193
MERCADO                               168
BAZAR                                 108
FERRAGEM                              104
MERCADINHO                             71
PET SHOP                               70
LOJA DE ROUPAS                         57
MINIMERCADO                            47
SAPATARIA                              46
L

In [17]:
# Todos os registros que contêm a palavra LOJA
candidatos_loja = other_ativos[
    other_ativos["descricao_classificacao"]
    .str.contains(
        r"\bLOJA\b",
        regex=True,
        na=False
    )
].copy()

print("Total contendo LOJA:")
print(len(candidatos_loja))

print("\n50 descrições com LOJA mais frequentes:")
print(
    candidatos_loja["descricao_classificacao"]
    .value_counts()
    .head(50)
    .to_string()
)

print("\n100 exemplos aleatórios contendo LOJA:")
print(
    candidatos_loja["descricao_classificacao"]
    .sample(
        min(100, len(candidatos_loja)),
        random_state=42
    )
    .tolist()
)

Total contendo LOJA:
1793

50 descrições com LOJA mais frequentes:
descricao_classificacao
LOJA                               194
LOJA DE ROUPAS                      57
LOJA DE MOVEIS                      33
LOJA DE INFORMATICA                 12
LOJA DE COLCHOES                    11
LOJA DE CALCADOS                    11
LOJA DE CONFECCOES                  10
LOJA DE VARIEDADES                   8
LOJA DE DECORACAO                    8
LOJA DE ROUPA                        7
LOJA DE AUTOMOVEIS                   7
LOJA ROUPAS                          7
LOJA DE ANTIGUIDADES                 6
LOJA COMERCIAL                       6
LOJA VEICULOS                        6
LOJA DE MATERIAIS DE CONSTRUCAO      6
LOJA DE FECHADURAS                   6
DEPOSITO LOJA                        5
DEPOSITO DE LOJA                     5
LOJA DE COSTURA                      5
LOJA DE ELETRONICOS                  5
LOJA DE VESTUARIO                    5
LOJA DE BEBIDAS                      5
LOJA DE ARTI

In [18]:
termos_inativos_pendentes = [
    r"\bFECHADOS\b",
    r"\bFECHADAS\b",
    r"\bA VENDA\b",
    r"\bPARA VENDA\b",
    r"\bPARA LOCACAO\b",
    r"\bP LOCACAO\b",
    r"\bP/ LOCACAO\b",
]

print("Ocorrências entre os registros ainda ativos:\n")

for padrao in termos_inativos_pendentes:
    mascara = (
        other_ativos["descricao_classificacao"]
        .str.contains(
            padrao,
            regex=True,
            na=False
        )
    )

    quantidade = mascara.sum()

    if quantidade > 0:
        print(f"{padrao}: {quantidade}")


padrao_conjunto = "|".join(termos_inativos_pendentes)

suspeitos = other_ativos[
    other_ativos["descricao_classificacao"]
    .str.contains(
        padrao_conjunto,
        regex=True,
        na=False
    )
].copy()

print("\nTotal encontrado:")
print(len(suspeitos))

print("\nDescrições encontradas:")
print(
    suspeitos["descricao_classificacao"]
    .value_counts()
    .head(60)
    .to_string()
)

Ocorrências entre os registros ainda ativos:

\bFECHADOS\b: 1
\bFECHADAS\b: 1
\bA VENDA\b: 15
\bPARA VENDA\b: 11
\bPARA LOCACAO\b: 17

Total encontrado:
45

Descrições encontradas:
descricao_classificacao
PARA LOCACAO                                 7
PARA VENDA                                   6
NAO IDENTIFICADO PARA LOCACAO                4
A VENDA                                      3
LOJA PARA LOCACAO                            3
PREDIO PARA LOCACAO                          1
ESCRITORIO COM GARAGEM A VENDA               1
NAO TEM NOME ESTA A VENDA                    1
3 LOJAS A VENDA                              1
CENTRO PROFISSIONAL MARIANTE PARA LOCACAO    1
SALA PARA LOCACAO                            1
LOJA E GARAGEM FECHADAS                      1
FECHADOS                                     1
GARAGEM A VENDA                              1
IMOVEL PARA VENDA                            1
PREDIO A VENDA                               1
FLORICULTURA PARA A VENDA DE FLORES        

In [19]:
# Termos que indicam que o próprio espaço/imóvel está disponível,
# e não que o estabelecimento vende algum produto.

tipos_imovel = (
    r"LOJA|LOJAS|"
    r"SALA|SALAS|"
    r"SALA COMERCIAL|SALAS COMERCIAIS|"
    r"PREDIO|"
    r"GARAGEM|"
    r"IMOVEL|"
    r"ESTABELECIMENTO|"
    r"ESPACO COMERCIAL|"
    r"PONTO COMERCIAL|"
    r"DEPOSITO|"
    r"ESCRITORIO|"
    r"GALPAO|"
    r"PAVILHAO|"
    r"CENTRO PROFISSIONAL"
)

padrao_imovel_disponivel = (
    rf"\b(?:{tipos_imovel})\b"
    rf".*"
    rf"\b(?:A VENDA|PARA VENDA|PARA LOCACAO)\b"
)

# Casos genéricos que também não representam atividade identificada
padrao_generico_disponivel = (
    r"^(?:"
    r"A VENDA|"
    r"PARA VENDA|"
    r"PARA LOCACAO|"
    r"FECHADOS|"
    r"FECHADAS"
    r")$"
)

# Casos sem identificação + disponibilidade
padrao_sem_identificacao_disponivel = (
    r"^(?:"
    r"NAO IDENTIFICADO.*PARA LOCACAO|"
    r"NAO TEM NOME.*A VENDA"
    r")$"
)

other_ativos["inativo_venda_locacao"] = (
    other_ativos["descricao_classificacao"]
    .str.contains(
        padrao_imovel_disponivel,
        regex=True,
        na=False
    )
    |
    other_ativos["descricao_classificacao"]
    .str.contains(
        padrao_generico_disponivel,
        regex=True,
        na=False
    )
    |
    other_ativos["descricao_classificacao"]
    .str.contains(
        padrao_sem_identificacao_disponivel,
        regex=True,
        na=False
    )
)

print("Registros marcados pela nova regra:")
print(
    other_ativos["inativo_venda_locacao"].sum()
)

print("\nDescrições marcadas:")
print(
    other_ativos.loc[
        other_ativos["inativo_venda_locacao"],
        "descricao_classificacao"
    ]
    .value_counts()
    .head(60)
    .to_string()
)

print("\nCasos com 'PARA VENDA' ou 'A VENDA' que NÃO foram marcados:")
nao_marcados = other_ativos[
    (
        other_ativos["descricao_classificacao"]
        .str.contains(
            r"\bPARA VENDA\b|\bA VENDA\b",
            regex=True,
            na=False
        )
    )
    &
    ~other_ativos["inativo_venda_locacao"]
]

print(
    nao_marcados["descricao_classificacao"]
    .value_counts()
    .head(40)
    .to_string()
)

Registros marcados pela nova regra:
39

Descrições marcadas:
descricao_classificacao
PARA LOCACAO                                 7
PARA VENDA                                   6
NAO IDENTIFICADO PARA LOCACAO                4
A VENDA                                      3
LOJA PARA LOCACAO                            3
PREDIO PARA LOCACAO                          1
ESCRITORIO COM GARAGEM A VENDA               1
NAO TEM NOME ESTA A VENDA                    1
3 LOJAS A VENDA                              1
CENTRO PROFISSIONAL MARIANTE PARA LOCACAO    1
SALA PARA LOCACAO                            1
FECHADOS                                     1
GARAGEM A VENDA                              1
IMOVEL PARA VENDA                            1
PREDIO A VENDA                               1
DEPOSITO PARA VENDA                          1
PREDIO VERDE A VENDA                         1
ESTABELECIMENTO A VENDA                      1
LOJA A VENDA                                 1
ESPACO COMERCIAL A VEN

In [20]:
# Acrescenta também FECHADOS / FECHADAS em qualquer posição da descrição
other_ativos["inativo_plural_fechado"] = (
    other_ativos["descricao_classificacao"]
    .str.contains(
        r"\bFECHADOS\b|\bFECHADAS\b",
        regex=True,
        na=False
    )
)

# Exclusões adicionais detectadas depois da regra v2
other_ativos["excluir_adicional"] = (
    other_ativos["inativo_venda_locacao"]
    | other_ativos["inativo_plural_fechado"]
)

print("Exclusões adicionais:")
print(other_ativos["excluir_adicional"].sum())

print("\nDescrições adicionais excluídas:")
print(
    other_ativos.loc[
        other_ativos["excluir_adicional"],
        "descricao_classificacao"
    ]
    .value_counts()
    .head(60)
    .to_string()
)

# Base final dos other_establishment utilizáveis
other_final = other_ativos[
    ~other_ativos["excluir_adicional"]
].copy()

print("\nRegistros antes das exclusões adicionais:")
print(len(other_ativos))

print("\nRegistros finais utilizáveis:")
print(len(other_final))

print("\nPercentual preservado dos 60.530 registros originais:")
print(
    round(
        len(other_final) / len(other) * 100,
        2
    )
)

Exclusões adicionais:
40

Descrições adicionais excluídas:
descricao_classificacao
PARA LOCACAO                                 7
PARA VENDA                                   6
NAO IDENTIFICADO PARA LOCACAO                4
A VENDA                                      3
LOJA PARA LOCACAO                            3
PREDIO PARA LOCACAO                          1
ESCRITORIO COM GARAGEM A VENDA               1
NAO TEM NOME ESTA A VENDA                    1
3 LOJAS A VENDA                              1
CENTRO PROFISSIONAL MARIANTE PARA LOCACAO    1
SALA PARA LOCACAO                            1
LOJA E GARAGEM FECHADAS                      1
FECHADOS                                     1
GARAGEM A VENDA                              1
IMOVEL PARA VENDA                            1
PREDIO A VENDA                               1
DEPOSITO PARA VENDA                          1
PREDIO VERDE A VENDA                         1
ESTABELECIMENTO A VENDA                      1
LOJA A VENDA            

In [21]:
# Termos de alta confiança para atividade de shopping/varejo
termos_shopping_forte = [
    r"\bMERCADO\b",
    r"\bMINI MERCADO\b",
    r"\bMINIMERCADO\b",
    r"\bMERCADINHO\b",
    r"\bSUPERMERCADO\b",
    r"\bSUPER MERCADO\b",
    r"\bHIPERMERCADO\b",

    r"\bBRECHO\b",
    r"\bBAZAR\b",
    r"\bARMARINHO\b",
    r"\bARMARINHOS\b",

    r"\bFERRAGEM\b",
    r"\bFERRAGENS\b",

    r"\bLIVRARIA\b",
    r"\bPAPELARIA\b",

    r"\bFLORICULTURA\b",

    r"\bSAPATARIA\b",
    r"\bCALCADOS\b",

    r"\bROUPA\b",
    r"\bROUPAS\b",
    r"\bVESTUARIO\b",
    r"\bCONFECCAO\b",
    r"\bCONFECCOES\b",

    r"\bMOVEIS\b",
    r"\bCOLCHOES\b",

    r"\bELETRODOMESTICOS\b",
    r"\bELETRONICOS\b",

    r"\bMATERIAL DE CONSTRUCAO\b",
    r"\bMATERIAIS DE CONSTRUCAO\b",

    r"\bPET SHOP\b",

    r"\bCOSMETICOS\b",
    r"\bPERFUMARIA\b",

    r"\bTINTAS\b",

    r"\bAUTO PECAS\b",
    r"\bAUTOPECAS\b",

    r"\bBICICLETAS\b",

    r"\bCELULAR\b",
    r"\bCELULARES\b",

    r"\bINFORMATICA\b",

    r"\bPRESENTES\b",
    r"\bVARIEDADES\b",

    r"\bANTIGUIDADES\b",

    r"\bTECIDOS\b",
    r"\bAVIAMENTOS\b",
]

padrao_shopping_forte = "|".join(
    termos_shopping_forte
)

other_final["shopping_forte"] = (
    other_final["descricao_classificacao"]
    .str.contains(
        padrao_shopping_forte,
        regex=True,
        na=False
    )
)

print("Total de registros utilizáveis:")
print(len(other_final))

print("\nClassificados como shopping de alta confiança:")
print(other_final["shopping_forte"].sum())

print("\nPercentual:")
print(
    round(
        other_final["shopping_forte"].mean() * 100,
        2
    )
)

print("\n40 descrições mais frequentes classificadas:")
print(
    other_final.loc[
        other_final["shopping_forte"],
        "descricao_classificacao"
    ]
    .value_counts()
    .head(40)
    .to_string()
)

print("\n50 exemplos aleatórios classificados:")
print(
    other_final.loc[
        other_final["shopping_forte"],
        "descricao_classificacao"
    ]
    .sample(
        min(
            50,
            other_final["shopping_forte"].sum()
        ),
        random_state=42
    )
    .tolist()
)

Total de registros utilizáveis:
54147

Classificados como shopping de alta confiança:
6070

Percentual:
11.21

40 descrições mais frequentes classificadas:
descricao_classificacao
MINI MERCADO                          211
BRECHO                                193
MERCADO                               168
BAZAR                                 108
FERRAGEM                              104
MERCADINHO                             71
PET SHOP                               70
LOJA DE ROUPAS                         57
MINIMERCADO                            47
SAPATARIA                              46
LOJA DE MOVEIS                         33
FLORICULTURA                           33
AUTO PECAS                             24
INFORMATICA                            21
SUPERMERCADO                           20
EMPRESA DE INFORMATICA                 13
CONFECCAO DE ROUPAS                    13
SUPER MERCADO                          12
LOJA DE INFORMATICA                    12
LOJA DE COLCHOES      

In [22]:
# Palavras que podem indicar serviço, fabricação ou manutenção,
# mesmo quando a descrição contém um produto comercial.

termos_nao_varejo = [
    r"\bOFICINA\b",
    r"\bFABRICA\b",
    r"\bFABRICACAO\b",
    r"\bINDUSTRIA\b",
    r"\bINDUSTRIAL\b",
    r"\bMANUTENCAO\b",
    r"\bCONSERTO\b",
    r"\bCONSERTOS\b",
    r"\bASSISTENCIA TECNICA\b",
    r"\bSERVICO\b",
    r"\bSERVICOS\b",
    r"\bPRODUCAO\b",
    r"\bMARCENARIA\b",
]

padrao_nao_varejo = "|".join(termos_nao_varejo)

shopping_suspeitos = other_final[
    other_final["shopping_forte"]
    &
    other_final["descricao_classificacao"]
    .str.contains(
        padrao_nao_varejo,
        regex=True,
        na=False
    )
].copy()

print("Total atualmente marcado como shopping_forte:")
print(other_final["shopping_forte"].sum())

print("\nShopping_forte com sinal de serviço/fabricação:")
print(len(shopping_suspeitos))

print("\nPercentual dos shopping_forte:")
print(
    round(
        len(shopping_suspeitos)
        / other_final["shopping_forte"].sum()
        * 100,
        2
    )
)

print("\n50 descrições suspeitas mais frequentes:")
print(
    shopping_suspeitos[
        "descricao_classificacao"
    ]
    .value_counts()
    .head(50)
    .to_string()
)

print("\n100 exemplos aleatórios:")
print(
    shopping_suspeitos[
        "descricao_classificacao"
    ]
    .sample(
        min(100, len(shopping_suspeitos)),
        random_state=42
    )
    .tolist()
)

Total atualmente marcado como shopping_forte:
6070

Shopping_forte com sinal de serviço/fabricação:
188

Percentual dos shopping_forte:
3.1

50 descrições suspeitas mais frequentes:
descricao_classificacao
FABRICA DE MOVEIS                                                                      10
FABRICA DE ROUPAS                                                                       4
OFICINA DE BICICLETAS                                                                   3
CONSERTO DE BICICLETAS                                                                  3
INDUSTRIA DE CONFECCAO                                                                  3
ASSISTENCIA TECNICA DE ELETRONICOS                                                      2
CONSERTO DE CELULARES                                                                   2
FABRICA DE TINTAS                                                                       2
FABRICACAO DE MOVEIS                                                      

In [23]:
# Termos que indicam que, apesar de haver serviço/fabricação,
# também existe atividade comercial ou venda de produtos.

termos_comercio_explicito = [
    r"\bCOMERCIO\b",
    r"\bLOJA\b",
    r"\bVENDA\b",
    r"\bVENDAS\b",
    r"\bPECAS\b",
    r"\bPRODUTO\b",
    r"\bPRODUTOS\b",
    r"\bACESSORIO\b",
    r"\bACESSORIOS\b",
    r"\bBRECHO\b",
    r"\bPET SHOP\b",
]

padrao_comercio_explicito = "|".join(
    termos_comercio_explicito
)

shopping_suspeitos["tem_comercio_explicito"] = (
    shopping_suspeitos["descricao_classificacao"]
    .str.contains(
        padrao_comercio_explicito,
        regex=True,
        na=False
    )
)

print("Total de casos suspeitos:")
print(len(shopping_suspeitos))

print("\nCom indício explícito de comércio/venda:")
print(
    shopping_suspeitos[
        "tem_comercio_explicito"
    ].sum()
)

print("\nSem indício explícito de comércio/venda:")
print(
    (~shopping_suspeitos[
        "tem_comercio_explicito"
    ]).sum()
)


print("\n--- COM comércio explícito ---")

print(
    shopping_suspeitos.loc[
        shopping_suspeitos[
            "tem_comercio_explicito"
        ],
        "descricao_classificacao"
    ]
    .value_counts()
    .head(50)
    .to_string()
)


print("\n--- SEM comércio explícito ---")

print(
    shopping_suspeitos.loc[
        ~shopping_suspeitos[
            "tem_comercio_explicito"
        ],
        "descricao_classificacao"
    ]
    .value_counts()
    .head(80)
    .to_string()
)

Total de casos suspeitos:
188

Com indício explícito de comércio/venda:
25

Sem indício explícito de comércio/venda:
163

--- COM comércio explícito ---
descricao_classificacao
FABRICA E COMERCIO DE MOVEIS                        2
LOJA DE CONSERTO DE CELULARES                       2
MAURO AUTO PECAS E SERVICOS                         1
LOJA DE FABRICA ROUPAS VINCO                        1
COMERCIO E CONSERTO DE ELETRODOMESTICOS             1
GAUCHO AUTO PECAS E SERVICOS                        1
GAUCHA AUTO PECAS E OFICINA                         1
LOJA DE ASSISTENCIA TECNICA DE CELULARES ROBERTO    1
SAPATARIA CONSERTO LOJA DE ROUPAS                   1
AUTO PECAS E OFICINA MECANICA FENIX                 1
LOJA DE SERVICOS ELETRONICOS                        1
PET SHOP PRODUTOS E SERVICOS VETERINARIO            1
LOJA CONSERTO CELULARES                             1
OFICINA E AUTO PECAS                                1
NO IMPACT AUTO PECAS E SERVICOS                     1
OFICINA DE CO

In [24]:
# Evidências mais fortes de que a atividade inclui efetivamente venda
termos_venda_forte = [
    r"\bCOMERCIO\b",
    r"\bVENDA\b",
    r"\bVENDAS\b",
    r"\bPECAS\b",
    r"\bPRODUTO\b",
    r"\bPRODUTOS\b",
    r"\bACESSORIO\b",
    r"\bACESSORIOS\b",
    r"\bBRECHO\b",
]

padrao_venda_forte = "|".join(
    termos_venda_forte
)

shopping_suspeitos["venda_forte"] = (
    shopping_suspeitos["descricao_classificacao"]
    .str.contains(
        padrao_venda_forte,
        regex=True,
        na=False
    )
)

print("Total de casos suspeitos:")
print(len(shopping_suspeitos))

print("\nCom evidência forte de venda:")
print(
    shopping_suspeitos["venda_forte"].sum()
)

print("\nSem evidência forte de venda:")
print(
    (~shopping_suspeitos["venda_forte"]).sum()
)


print("\n--- COM evidência forte de venda ---")
print(
    shopping_suspeitos.loc[
        shopping_suspeitos["venda_forte"],
        "descricao_classificacao"
    ]
    .value_counts()
    .to_string()
)


print("\n--- SEM evidência forte de venda ---")
print(
    shopping_suspeitos.loc[
        ~shopping_suspeitos["venda_forte"],
        "descricao_classificacao"
    ]
    .value_counts()
    .head(80)
    .to_string()
)

Total de casos suspeitos:
188

Com evidência forte de venda:
18

Sem evidência forte de venda:
170

--- COM evidência forte de venda ---
descricao_classificacao
FABRICA E COMERCIO DE MOVEIS                        2
MAURO AUTO PECAS E SERVICOS                         1
COMERCIO E CONSERTO DE ELETRODOMESTICOS             1
GAUCHO AUTO PECAS E SERVICOS                        1
GAUCHA AUTO PECAS E OFICINA                         1
AUTO PECAS E OFICINA MECANICA FENIX                 1
PET SHOP PRODUTOS E SERVICOS VETERINARIO            1
OFICINA E AUTO PECAS                                1
NO IMPACT AUTO PECAS E SERVICOS                     1
OFICINA DE COSTURA E BRECHO                         1
CELULAR E POINT ASSISTENCIA TECNICA E ACESSORIOS    1
LOJA DE INFORMATICA VENDAS E MANUTENCAO             1
M BIKE BICICLETAS OFICINA E PECAS                   1
GAUCHA AUTO PECAS E OFICINA MECANICA                1
OFICINA MECANICA HG AUTO PECAS                      1
JOSEFINA BRECHO CONSERTOS E R

Loja

In [25]:
# Mantém shopping_forte quando:
# 1. não há sinais de serviço/fabricação
# OU
# 2. há sinais de serviço/fabricação, mas também há evidência forte de venda

indices_suspeitos = shopping_suspeitos.index

other_final["shopping_v1"] = other_final["shopping_forte"]

# Remove temporariamente todos os suspeitos
other_final.loc[
    indices_suspeitos,
    "shopping_v1"
] = False

# Reinsere apenas os suspeitos com evidência forte de venda
indices_resgatados = shopping_suspeitos.loc[
    shopping_suspeitos["venda_forte"]
].index

other_final.loc[
    indices_resgatados,
    "shopping_v1"
] = True


print("Shopping_forte original:")
print(other_final["shopping_forte"].sum())

print("\nShopping_v1 após refinamento:")
print(other_final["shopping_v1"].sum())

print("\nRemovidos por serem serviço/fabricação sem venda explícita:")
print(
    (
        other_final["shopping_forte"]
        & ~other_final["shopping_v1"]
    ).sum()
)

print("\nCasos resgatados por terem venda explícita:")
print(len(indices_resgatados))

print("\n30 exemplos removidos de shopping:")
print(
    other_final.loc[
        other_final["shopping_forte"]
        & ~other_final["shopping_v1"],
        "descricao_classificacao"
    ]
    .value_counts()
    .head(30)
    .to_string()
)

print("\n30 exemplos mantidos como shopping_v1:")
print(
    other_final.loc[
        other_final["shopping_v1"],
        "descricao_classificacao"
    ]
    .sample(
        min(
            30,
            other_final["shopping_v1"].sum()
        ),
        random_state=42
    )
    .tolist()
)

Shopping_forte original:
6070

Shopping_v1 após refinamento:
5900

Removidos por serem serviço/fabricação sem venda explícita:
170

Casos resgatados por terem venda explícita:
18

30 exemplos removidos de shopping:
descricao_classificacao
FABRICA DE MOVEIS                                                                      10
FABRICA DE ROUPAS                                                                       4
OFICINA DE BICICLETAS                                                                   3
CONSERTO DE BICICLETAS                                                                  3
INDUSTRIA DE CONFECCAO                                                                  3
ASSISTENCIA TECNICA DE ELETRONICOS                                                      2
CONSERTO DE CELULARES                                                                   2
FABRICA DE TINTAS                                                                       2
FABRICACAO DE MOVEIS                     

In [26]:
# Registros que contêm LOJA,
# mas ainda NÃO foram classificados como shopping_v1

lojas_pendentes = other_final[
    (
        other_final["descricao_classificacao"]
        .str.contains(
            r"\bLOJA\b",
            regex=True,
            na=False
        )
    )
    &
    ~other_final["shopping_v1"]
].copy()


print("Total contendo LOJA:")
print(
    other_final["descricao_classificacao"]
    .str.contains(
        r"\bLOJA\b",
        regex=True,
        na=False
    )
    .sum()
)

print("\nJá classificados como shopping_v1:")
print(
    (
        other_final["shopping_v1"]
        &
        other_final["descricao_classificacao"]
        .str.contains(
            r"\bLOJA\b",
            regex=True,
            na=False
        )
    ).sum()
)

print("\nLOJAS ainda pendentes:")
print(len(lojas_pendentes))


print("\nDescrições mais frequentes entre as pendentes:")
print(
    lojas_pendentes[
        "descricao_classificacao"
    ]
    .value_counts()
    .head(60)
    .to_string()
)


print("\n100 exemplos aleatórios:")
print(
    lojas_pendentes[
        "descricao_classificacao"
    ]
    .sample(
        min(100, len(lojas_pendentes)),
        random_state=42
    )
    .tolist()
)

Total contendo LOJA:
1788

Já classificados como shopping_v1:
630

LOJAS ainda pendentes:
1158

Descrições mais frequentes entre as pendentes:
descricao_classificacao
LOJA                                          194
LOJA DE DECORACAO                               8
LOJA DE AUTOMOVEIS                              7
LOJA COMERCIAL                                  6
LOJA VEICULOS                                   6
LOJA DE FECHADURAS                              6
DEPOSITO LOJA                                   5
DEPOSITO DE LOJA                                5
LOJA DE COSTURA                                 5
LOJA DE BEBIDAS                                 5
LOJA DE ARTIGOS RELIGIOSOS                      4
LOJA DE AGROPECUARIA                            4
LOJA PET                                        4
LOJA ORTOBOM                                    4
LOJA DE BORDADOS                                4
HELP LOJA DE CREDITO                            4
LOJA DE BICICLETA                

In [27]:
# Termos que tornam a classificação como shopping duvidosa.
# Por enquanto servem APENAS para auditoria.

termos_loja_suspeita = [
    r"\bCONSERTO\b",
    r"\bCONSERTOS\b",
    r"\bMANUTENCAO\b",
    r"\bASSISTENCIA TECNICA\b",
    r"\bSERVICO\b",
    r"\bSERVICOS\b",
    r"\bOFICINA\b",

    r"\bSEGURO\b",
    r"\bSEGUROS\b",
    r"\bCREDITO\b",
    r"\bFINANCEIRA\b",

    r"\bCOSTURA\b",
    r"\bBARBEARIA\b",
    r"\bSALAO DE BELEZA\b",

    r"\bDEPOSITO\b",
    r"\bFABRICA\b",

    r"\bDESOCUPADA\b",
    r"\bDESOCUPADO\b",
    r"\bDEDOCUPADA\b",   # erro de digitação presente na base
    r"\bSEM USO\b",
    r"\bEM CONSTRUCAO\b",
]

padrao_loja_suspeita = "|".join(
    termos_loja_suspeita
)

lojas_pendentes["loja_suspeita"] = (
    lojas_pendentes["descricao_classificacao"]
    .str.contains(
        padrao_loja_suspeita,
        regex=True,
        na=False
    )
)

print("Total de lojas pendentes:")
print(len(lojas_pendentes))

print("\nCom algum sinal de serviço/apoio/inatividade:")
print(
    lojas_pendentes["loja_suspeita"].sum()
)

print("\nSem esses sinais:")
print(
    (~lojas_pendentes["loja_suspeita"]).sum()
)


print("\nOcorrências por termo:\n")

for termo in termos_loja_suspeita:
    quantidade = (
        lojas_pendentes["descricao_classificacao"]
        .str.contains(
            termo,
            regex=True,
            na=False
        )
        .sum()
    )

    if quantidade > 0:
        print(f"{termo}: {quantidade}")


print("\n--- LOJAS SUSPEITAS ---")
print(
    lojas_pendentes.loc[
        lojas_pendentes["loja_suspeita"],
        "descricao_classificacao"
    ]
    .value_counts()
    .head(80)
    .to_string()
)


print("\n--- AMOSTRA DAS LOJAS SEM SINAIS SUSPEITOS ---")
print(
    lojas_pendentes.loc[
        ~lojas_pendentes["loja_suspeita"],
        "descricao_classificacao"
    ]
    .sample(
        min(
            80,
            (~lojas_pendentes["loja_suspeita"]).sum()
        ),
        random_state=42
    )
    .tolist()
)

Total de lojas pendentes:
1158

Com algum sinal de serviço/apoio/inatividade:
87

Sem esses sinais:
1071

Ocorrências por termo:

\bCONSERTO\b: 9
\bCONSERTOS\b: 4
\bMANUTENCAO\b: 5
\bASSISTENCIA TECNICA\b: 5
\bSERVICOS\b: 4
\bOFICINA\b: 3
\bSEGUROS\b: 1
\bCREDITO\b: 7
\bCOSTURA\b: 10
\bBARBEARIA\b: 2
\bSALAO DE BELEZA\b: 3
\bDEPOSITO\b: 27
\bFABRICA\b: 3
\bDEDOCUPADA\b: 2
\bSEM USO\b: 3
\bEM CONSTRUCAO\b: 1

--- LOJAS SUSPEITAS ---
descricao_classificacao
DEPOSITO LOJA                                       5
DEPOSITO DE LOJA                                    5
LOJA DE COSTURA                                     5
HELP LOJA DE CREDITO                                4
LOJA SEM USO NO MOMENTO                             3
LOJA DEDOCUPADA                                     2
LOJA DE CONSERTO DE CELULARES                       2
LOJA MANUTENCAO PREDIAL                             2
LOJA DE CREDITO AGIBANK                             1
LOJA E BARBEARIA                                    1


In [28]:
lojas_candidatas_varejo = lojas_pendentes[
    ~lojas_pendentes["loja_suspeita"]
].copy()

termos_servico_loja = [
    r"\bBANHO E TOSA\b",
    r"\bTOSA\b",
    r"\bBELEZA\b",
    r"\bESTETICA\b",
    r"\bCABELEIREIRO\b",
    r"\bCABELEREIRO\b",
    r"\bMECANICA\b",
    r"\bLAVAGEM\b",
    r"\bLAVANDERIA\b",
    r"\bIMOBILIARIA\b",
    r"\bADVOCACIA\b",
    r"\bCONTABILIDADE\b",
    r"\bCONSULTORIA\b",
    r"\bDESPACHANTE\b",
    r"\bXEROX\b",
    r"\bCOPIAS\b",
    r"\bFOTOGRAFIA\b",
    r"\bFOTOGRAFICO\b",
    r"\bLOCADORA\b",
    r"\bALUGUEL\b",
    r"\bLOCACAO\b",
    r"\bESTACIONAMENTO\b",
]

padrao_servico_loja = "|".join(
    termos_servico_loja
)

lojas_candidatas_varejo["servico_oculto"] = (
    lojas_candidatas_varejo["descricao_classificacao"]
    .str.contains(
        padrao_servico_loja,
        regex=True,
        na=False
    )
)

print("Lojas candidatas a varejo:")
print(len(lojas_candidatas_varejo))

print("\nCom algum novo sinal de serviço:")
print(
    lojas_candidatas_varejo["servico_oculto"].sum()
)

print("\nOcorrências por termo:\n")

for termo in termos_servico_loja:
    quantidade = (
        lojas_candidatas_varejo["descricao_classificacao"]
        .str.contains(
            termo,
            regex=True,
            na=False
        )
        .sum()
    )

    if quantidade > 0:
        print(f"{termo}: {quantidade}")

print("\nDescrições encontradas:")
print(
    lojas_candidatas_varejo.loc[
        lojas_candidatas_varejo["servico_oculto"],
        "descricao_classificacao"
    ]
    .value_counts()
    .head(80)
    .to_string()
)

Lojas candidatas a varejo:
1071

Com algum novo sinal de serviço:
22

Ocorrências por termo:

\bBANHO E TOSA\b: 1
\bTOSA\b: 1
\bBELEZA\b: 2
\bESTETICA\b: 4
\bMECANICA\b: 3
\bLAVANDERIA\b: 3
\bXEROX\b: 2
\bFOTOGRAFICO\b: 2
\bALUGUEL\b: 3
\bESTACIONAMENTO\b: 2

Descrições encontradas:
descricao_classificacao
LOJA DE MATERIAL FOTOGRAFICO           2
LOJA DE ESTETICA                       1
LOJA 2 STUDIO FLOR DE LIZ ESTETICA     1
LOJA DE ALUGUEL DE FANTASIAS           1
LOJA DE BANHO E TOSA CANINA            1
LOJA E SALAO BELEZA HELENITA           1
GTR MECANICA LOJA 1                    1
LOJA E ESTETICA KRIATIVA               1
LOJA ESTACAO DA BELEZA                 1
LOJA DE XEROX                          1
LOJA XEROX                             1
MECANICA LOJA                          1
LOJA MECANICA                          1
LOJA PARA ALUGUEL                      1
ESTACIONAMENTO DA LOJA DE PLASTICOS    1
ESTETICA E LOJA MARINOS                1
ESTACIONAMENTO DA LOJA DE TECIDO    

In [29]:
# Casos que parecem serviço/locação/apoio e não devem entrar
# automaticamente como shopping.

# Exceção claramente varejista dentro do grupo auditado
excecoes_varejo = {
    "LOJA DE MATERIAL FOTOGRAFICO",
}

lojas_candidatas_varejo["shopping_loja_valida"] = (
    ~lojas_candidatas_varejo["servico_oculto"]
    |
    lojas_candidatas_varejo[
        "descricao_classificacao"
    ].isin(excecoes_varejo)
)

print("Total de lojas candidatas:")
print(len(lojas_candidatas_varejo))

print("\nLojas aceitas como shopping:")
print(
    lojas_candidatas_varejo[
        "shopping_loja_valida"
    ].sum()
)

print("\nLojas rejeitadas nesta etapa:")
print(
    (~lojas_candidatas_varejo[
        "shopping_loja_valida"
    ]).sum()
)

print("\nDescrições rejeitadas:")
print(
    lojas_candidatas_varejo.loc[
        ~lojas_candidatas_varejo[
            "shopping_loja_valida"
        ],
        "descricao_classificacao"
    ]
    .value_counts()
    .to_string()
)

print("\nExceções preservadas como varejo:")
print(
    lojas_candidatas_varejo.loc[
        lojas_candidatas_varejo[
            "descricao_classificacao"
        ].isin(excecoes_varejo),
        "descricao_classificacao"
    ]
    .value_counts()
    .to_string()
)

Total de lojas candidatas:
1071

Lojas aceitas como shopping:
1051

Lojas rejeitadas nesta etapa:
20

Descrições rejeitadas:
descricao_classificacao
LOJA DE ESTETICA                       1
LOJA 2 STUDIO FLOR DE LIZ ESTETICA     1
LOJA DE ALUGUEL DE FANTASIAS           1
LOJA DE BANHO E TOSA CANINA            1
LOJA E SALAO BELEZA HELENITA           1
GTR MECANICA LOJA 1                    1
LOJA E ESTETICA KRIATIVA               1
LOJA ESTACAO DA BELEZA                 1
LOJA DE XEROX                          1
LOJA XEROX                             1
MECANICA LOJA                          1
LOJA MECANICA                          1
LOJA PARA ALUGUEL                      1
ESTACIONAMENTO DA LOJA DE PLASTICOS    1
ESTETICA E LOJA MARINOS                1
ESTACIONAMENTO DA LOJA DE TECIDO       1
LAVANDERIA LOJA 3                      1
LOJA DE ALUGUEL DE CARROS VGN GROUP    1
LOJA DE LAVANDERIA                     1
LA MAMA LOJA E LAVANDERIA              1

Exceções preservadas como vare

In [30]:
# Parte da classificação de alta confiança já existente
other_final["shopping_v2"] = (
    other_final["shopping_v1"].copy()
)

# Índices das lojas adicionais aceitas
indices_lojas_validas = (
    lojas_candidatas_varejo.loc[
        lojas_candidatas_varejo["shopping_loja_valida"]
    ].index
)

# Incorpora essas lojas ao shopping
other_final.loc[
    indices_lojas_validas,
    "shopping_v2"
] = True


print("Shopping_v1:")
print(
    other_final["shopping_v1"].sum()
)

print("\nLojas adicionais incorporadas:")
print(
    len(indices_lojas_validas)
)

print("\nShopping_v2:")
print(
    other_final["shopping_v2"].sum()
)

print("\nPercentual dos other_establishment utilizáveis:")
print(
    round(
        other_final["shopping_v2"].mean() * 100,
        2
    )
)

print("\nNovos casos adicionados em relação ao shopping_v1:")
print(
    (
        other_final["shopping_v2"]
        & ~other_final["shopping_v1"]
    ).sum()
)

print("\n40 exemplos aleatórios do shopping_v2:")
print(
    other_final.loc[
        other_final["shopping_v2"],
        "descricao_classificacao"
    ]
    .sample(
        min(
            40,
            other_final["shopping_v2"].sum()
        ),
        random_state=42
    )
    .tolist()
)

Shopping_v1:
5900

Lojas adicionais incorporadas:
1051

Shopping_v2:
6951

Percentual dos other_establishment utilizáveis:
12.84

Novos casos adicionados em relação ao shopping_v1:
1051

40 exemplos aleatórios do shopping_v2:
['MINI MERCADO PROTASIO ALVES', 'LOJA E CAFETERIA', 'SAL COMERCIO DE MOVEIS', 'LOJA DE MOTOS', 'GEORGIANA LOJA DE ROUPA', 'FERRAGEM DAQUI', 'MINI MERCADO GRIZOTTI', 'BAZAR', 'LOJA MUNAY', 'ESTACIONAMENTO E LAVA RAPIDO MR CAR E PET SHOP OURO PETS', 'RAKON CALCADOS', 'BRECHO SOLIDARIO', 'CONFECCAO DE UNIFORMES', 'BRECHO', 'MINIMERCADO PESSI', 'LOJA DE MOVEIS E BAZAR CASA BONITA', 'BRECHO CAMINHO DOS ANTIQUARIOS', 'PET SHOP PET STILO', 'MINIMERCADO ANDORINHA', 'BRECHO E SEU MEU BEM', 'FERRAGEM SANTANA', 'CONFECCAO NETTE VARIEDADES', 'MOVEIS SANTA RITA OUTLET', 'TRAJE PRIME LOJA DE SURFE', 'MINI MERCADO KASTELAO', 'LU PRESENTES', 'ARMAZEM MERCADO DUARTE', 'MERCADO DOS CAES PECUARIA E FERRAGEM', 'BRECHO', 'ESTILO MOVEIS', 'PET SHOP', 'BOUTIQUE DE ROUPAS', 'VETERINARIA 